# 03 - concatenating the data

two steps here. first we stack all the monthly parquet files for each cab type into one big file per type. then we combine yellow, green and hvfhv into a single concatenated dataset.

both steps use duckdb which handles the merging at the file level without loading everything into memory at once.

## step 1 - concat monthly files per cab type

the raw data comes as one parquet per month per cab type. this merges them into `data/yellow.parquet`, `data/green.parquet` etc. using a glob pattern so duckdb handles all the files in one go.

In [ ]:
import os
import duckdb

RAW_DIR = os.path.join(os.path.dirname(os.getcwd()), 'raw')
OUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
os.makedirs(OUT_DIR, exist_ok=True)

TYPES = ['fhv', 'yellow', 'fhvhv', 'green']

con = duckdb.connect()

for cab in TYPES:
    folder   = os.path.join(RAW_DIR, cab)
    out_path = os.path.join(OUT_DIR, f'{cab}.parquet')

    if not os.path.exists(folder):
        print(f'{cab}: folder not found, skipping')
        continue

    pattern = os.path.join(folder, '*.parquet')
    print(f'merging {cab}...')

    try:
        con.execute(f"""
            COPY (
                SELECT * FROM read_parquet('{pattern}', union_by_name=True)
            ) TO '{out_path}' (FORMAT PARQUET)
        """)
        row_count = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{out_path}')"
        ).fetchone()[0]
        print(f'  {cab}: {row_count:,} rows -> {out_path}')
    except Exception as e:
        print(f'  {cab}: failed ({e})')

## step 2 - combine cleaned cab types

now we take the cleaned outputs from the previous notebook (clean_yellow, clean_green, clean_fhvhv) and stack them into one combined parquet. we use `union_by_name=True` so any columns that don't exist in a given type get filled with nulls.

In [ ]:
import os
import duckdb

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
OUT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'combined', 'post_clean_concat_30_bucket.parquet')

TYPES = ['fhv', 'yellow', 'green']

con = duckdb.connect()

files = [os.path.join(DATA_DIR, f'clean_{cab}.parquet') for cab in TYPES]

for f in files:
    if not os.path.exists(f):
        print(f'missing: {f}, aborting')
        raise FileNotFoundError(f)

pattern = "', '".join(files)
print('combining all parquet files...')

con.execute(f"""
    COPY (
        SELECT * FROM read_parquet(['{pattern}'], union_by_name=True)
    ) TO '{OUT_PATH}' (FORMAT PARQUET)
""")

row_count = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{OUT_PATH}')"
).fetchone()[0]
print(f'done: {row_count:,} rows -> {OUT_PATH}')